# Linear ridge regression example for the challenge Dyni Odontocete Click Classification, 10 species

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
import time

In [2]:
# read data
X_train = np.load('./DOCC10_train/DOCC10_Xtrain.npy')
Y_train_df = pd.read_csv('./labels.csv', index_col=0)


print('X train', X_train.shape, 'Y_train', Y_train_df.shape)
Y_train_df.describe()

X train (113120, 8192) Y_train (113120, 1)


,TARGET
count,113120
unique,10
top,UDA
freq,11312


In [4]:
print(X_train.shape, X_train.dtype)
print(X_train[:10])

(113120, 8192) float32
[[-1.0681152e-03 -1.0986328e-03 -1.0070801e-03 ... -3.6621094e-04
  -6.1035156e-05  0.0000000e+00]
 [ 2.4108887e-03  1.9226074e-03  2.6550293e-03 ... -2.4414062e-04
   3.0517578e-04  9.1552734e-05]
 [ 1.1901855e-03  1.5869141e-03  1.2207031e-03 ... -6.1035156e-05
  -9.1552734e-04 -5.4931641e-04]
 ...
 [ 7.6293945e-04  4.5776367e-04  7.9345703e-04 ...  8.5449219e-04
   3.3569336e-04  2.4414062e-04]
 [ 1.5258789e-03  1.7395020e-03  1.7395020e-03 ...  7.0190430e-04
   6.4086914e-04  7.6293945e-04]
 [ 1.2207031e-03  1.3427734e-03  1.5869141e-03 ... -8.5449219e-04
  -1.1291504e-03 -1.0681152e-03]]


In [ ]:
# encorde string labels with one-hot vectors
Y_tr_ = Y_train_df['TARGET'].values.reshape(-1, 1)
enc = OneHotEncoder()
enc.fit(Y_tr_)
Y_tr = enc.transform(Y_tr_).toarray()

In [ ]:
# shuffle train set
perm = np.random.permutation(X_train.shape[0])
X = X_train[perm]
y = Y_tr[perm]
X.shape, y.shape

In [ ]:
# cross validate the ridge parameter alpha
alphas = 10.**(-np.arange(-5,5))
best_acc = 0
best_alpha = -1

for alpha in alphas:
    t0 = time.time()
    reg = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    y_pred = cross_val_predict(reg, X=X, y=y, cv=4)
    acc = accuracy_score(np.argmax(y, axis=1), np.argmax(y_pred, axis=1))
    if acc > best_acc:
        best_acc, best_alpha = acc, alpha
    t = (time.time() - t0) / 60
    print(f'alpha={alpha} acc={acc}, t = {t:.1f} min')

In [ ]:
# fit regressor on the whole train db
reg = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha))
reg.fit(X, y)

In [ ]:
# save predictions to csv for submission
y_test_pred = reg.predict(X_test.astype('float32'))
y_test_pred = (y_test_pred == y_test_pred.max(axis=1, keepdims=True)).astype(float)
y_test_pred = enc.inverse_transform(y_test_pred)
Y_test_pred_df = pd.DataFrame(data=y_test_pred, index=np.arange(y_test_pred.shape[0]), columns=['TARGET'])
Y_test_pred_df.to_csv('./DOCC10_Ytest_pred_linear.csv', index_label='ID')